# Production Analytics and ML

Notebook для этапов очистки, анализа, ML и визуализации.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

pg_uri = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'etl_user')}:{os.getenv('POSTGRES_PASSWORD', 'etl_pass')}"
    f"@{os.getenv('POSTGRES_HOST', 'postgres')}:{os.getenv('POSTGRES_PORT', '5432')}/{os.getenv('POSTGRES_DB', 'oil_analytics')}"
)
engine = create_engine(pg_uri)

In [ ]:
daily = pd.read_sql('SELECT * FROM mart.daily_production ORDER BY date', engine)
kpi = pd.read_sql('SELECT * FROM mart.well_kpi ORDER BY avg_oil_ton DESC', engine)
influence = pd.read_sql('SELECT * FROM mart.influence_factors', engine)
metrics = pd.read_sql('SELECT * FROM mart.ml_metrics', engine)

daily.head(), kpi.head(), influence, metrics

In [ ]:
plt.figure(figsize=(10, 4))
sns.lineplot(data=daily, x='date', y='total_oil_ton')
plt.title('Daily Oil Production')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=kpi.head(10), x='name', y='avg_oil_ton')
plt.title('Top Wells by Average Oil Ton')
plt.xticks(rotation=45)
plt.tight_layout()

In [ ]:
features = pd.read_sql('SELECT avg_pressure, avg_temperature, oil_ton FROM mart.well_kpi k JOIN production p USING (well_id) LIMIT 1000', engine)
pivot = features.pivot_table(values='oil_ton', index='avg_temperature', columns='avg_pressure', aggfunc='mean')
plt.figure(figsize=(8, 6))
sns.heatmap(pivot, cmap='viridis')
plt.title('Pressure vs Temperature Impact on Oil')
plt.tight_layout()